# 06 Feature Redundancy Audit

목적: `06_significance_tests_and_eda.ipynb`에서 선별한 모델링 후보 변수들이 서로 과도하게 중복되는지 점검합니다.

이 노트북은 07번 모델링을 직접 수행하지 않습니다. 대신 07번으로 넘기기 전, 후보 변수 간 고상관, 구조적 중복, VIF 위험을 감사하고, 어떤 변수를 모델 후보로 유지/주의/검토할지 기록합니다.

핵심 원칙:

- 입력 원천은 05번 최종 모델링 테이블과 최신 06번 후보 변수 파일입니다.
- `_data`에는 쓰지 않습니다.
- 산출물은 `park.ingyeom/reports/data/06_feature_redundancy_audit`와 `park.ingyeom/reports/tables/06_feature_redundancy_audit`에 저장합니다.
- 다중공선성은 “변수를 자동 삭제하는 판정”이 아니라, 07번 모델링 전 해석 위험을 표시하는 감사 단계로 다룹니다.
- VIF만으로는 “어떤 변수와 어떤 변수 사이에서 중복이 발생하는지”가 바로 보이지 않기 때문에, 고상관 pair / 구조적 duplicate / 클러스터 / VIF 관련 변수표를 함께 남깁니다.

In [15]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

## 1. 실행 옵션

기본값은 재현성과 검산을 우선합니다. VIF와 상관관계 계산은 06번 후보 변수와 구조적 중복 위험이 큰 core 변수 중심으로 수행합니다.

In [16]:
NOTEBOOK_ID = "06_feature_redundancy_audit"

TARGET_COL = "is_repurchase"
ID_COL = "membership_row_id"

CORR_METHOD = "spearman"
HIGH_CORR_THRESHOLD = 0.85
NEAR_DUPLICATE_CORR_THRESHOLD = 0.98
MIN_CORR_NON_NULL_N = 100
MIN_NON_NULL_RATIO = 0.50
MAX_MISSING_RATIO_FOR_VIF = 0.30
VIF_WARN_THRESHOLD = 5.0
VIF_HIGH_THRESHOLD = 10.0
VIF_INF_R2_THRESHOLD = 0.999999

RUN_VIF = True
RUN_FIGURES = True
SHOW_TABLES = False
FIG_DPI = 200
MAX_HEATMAP_FEATURES = 24
MAX_RELATED_PER_FEATURE = 5
TOP_PAIR_DETAIL_COUNT = 6

# 06 후보 파일에 없더라도 구조적 중복 위험이 큰 변수는 같이 감사합니다.
FORCE_AUDIT_FEATURES = [
    "is_100won", "is_promotion", "price",
    "max_screen", "screen_1_flag", "screen_2_flag", "screen_4_flag",
    "promo_x_1screen", "promo_x_2screen", "promo_x_4screen",
    "has_watch_obs", "no_watch_obs_flag", "has_usage_feature", "no_usage_feature_flag",
    "has_content_feature", "no_content_feature_flag",
    "metadata_covered_watch_ratio", "metadata_missing_watch_ratio", "usable_metadata_watch_ratio",
    "week1_ratio", "week2_ratio", "week3_ratio", "front_loaded_ratio", "late_ratio",
    "total_watch_time", "week1_watch_time", "week2_watch_time", "week3_watch_time",
]

## 2. 경로 설정

01~06 본체와 같은 원칙을 따릅니다. 프로젝트 루트는 `.git`과 `_data`가 있는 저장소 루트이고, 개인 산출물은 `park.ingyeom/reports` 아래에 저장합니다.

In [17]:
def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / ".git").exists() and (candidate / "_data").exists():
            return candidate

    for candidate in candidates:
        if candidate.name == "park.ingyeom" and (candidate.parent / "_data").exists():
            return candidate.parent

    raise FileNotFoundError("저장소 루트를 찾지 못했습니다. C:\\Code\\ott-churn-prediction 또는 그 하위에서 실행하세요.")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "_data"
WORK_ROOT = PROJECT_ROOT / "park.ingyeom"
REPORTS_DIR = WORK_ROOT / "reports"

INPUT_05_DATA_DIR = REPORTS_DIR / "data" / "05_content_feature_engineering"
INPUT_06_DATA_DIR = REPORTS_DIR / "data" / "06_significance_tests_and_eda"
INPUT_06_TABLE_DIR = REPORTS_DIR / "tables" / "06_significance_tests_and_eda"

OUTPUT_DATA_DIR = REPORTS_DIR / "data" / NOTEBOOK_ID
OUTPUT_TABLE_DIR = REPORTS_DIR / "tables" / NOTEBOOK_ID
OUTPUT_FIGURE_DIR = REPORTS_DIR / "figures" / NOTEBOOK_ID

for d in [OUTPUT_DATA_DIR, OUTPUT_TABLE_DIR, OUTPUT_FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_05_DATA_DIR:", INPUT_05_DATA_DIR)
print("INPUT_06_DATA_DIR:", INPUT_06_DATA_DIR)
print("INPUT_06_TABLE_DIR:", INPUT_06_TABLE_DIR)
print("OUTPUT_DATA_DIR:", OUTPUT_DATA_DIR)
print("OUTPUT_TABLE_DIR:", OUTPUT_TABLE_DIR)
print("OUTPUT_FIGURE_DIR:", OUTPUT_FIGURE_DIR)

PROJECT_ROOT: c:\Code\ott-churn-prediction
INPUT_05_DATA_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\05_content_feature_engineering
INPUT_06_DATA_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\06_significance_tests_and_eda
INPUT_06_TABLE_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\tables\06_significance_tests_and_eda
OUTPUT_DATA_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\06_feature_redundancy_audit
OUTPUT_TABLE_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\tables\06_feature_redundancy_audit
OUTPUT_FIGURE_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\figures\06_feature_redundancy_audit


## 3. 한글 폰트 설정

시각화에서 한글이 깨지지 않도록 가능한 폰트를 탐색합니다. Windows에서는 `Malgun Gothic`, macOS에서는 `AppleGothic`, 그 외 환경에서는 `NanumGothic`, `Noto Sans KR` 계열을 우선 탐색합니다.

In [18]:
def configure_korean_font() -> dict:
    preferred_fonts = [
        "Malgun Gothic",
        "AppleGothic",
        "NanumGothic",
        "Nanum Gothic",
        "Noto Sans CJK KR",
        "Noto Sans KR",
        "Noto Sans CJK",
        "Arial Unicode MS",
        "DejaVu Sans",
    ]

    available_names = {f.name for f in font_manager.fontManager.ttflist}
    selected = None
    for name in preferred_fonts:
        if name in available_names:
            selected = name
            break

    if selected is None:
        selected = plt.rcParams.get("font.family", ["sans-serif"])
        if isinstance(selected, list):
            selected = selected[0]

    plt.rcParams["font.family"] = selected
    plt.rcParams["axes.unicode_minus"] = False

    return {
        "selected_font": selected,
        "preferred_fonts": preferred_fonts,
        "available_font_count": len(available_names),
    }

font_config = configure_korean_font()
font_config_df = pd.DataFrame([
    {"metric": "selected_font", "value": font_config["selected_font"]},
    {"metric": "available_font_count", "value": font_config["available_font_count"]},
    {"metric": "preferred_font_candidates", "value": ", ".join(font_config["preferred_fonts"])},
])
font_config_df.to_csv(OUTPUT_TABLE_DIR / "06_feature_redundancy_font_config.csv", index=False, encoding="utf-8-sig")
display(font_config_df)

,metric,value
0,selected_font,Malgun Gothic
1,available_font_count,204
2,preferred_font_candidates,"Malgun Gothic, AppleGothic, NanumGothic, Nanum..."


## 4. 입력 파일 로드 및 검산

최신 구조는 `reports/data/06_significance_tests_and_eda`를 기준으로 합니다. 다만 과거 실행 산출물이 tables에만 있는 경우를 진단하기 위해 06 결과 파일은 data를 우선하고 tables를 fallback으로 봅니다.

In [19]:
def first_existing(candidates: Iterable[Path], label: str, required: bool = True) -> Path | None:
    candidates = list(candidates)
    for p in candidates:
        if p.exists():
            return p
    if required:
        msg = "\n".join(str(p) for p in candidates)
        raise FileNotFoundError(f"{label} 파일을 찾지 못했습니다. 후보 경로:\n{msg}")
    return None

PATH_MODELING_TABLE = first_existing([
    INPUT_05_DATA_DIR / "modeling_feature_table_with_content.csv",
], "05 modeling_feature_table_with_content.csv")

PATH_CANDIDATE_FEATURES = first_existing([
    INPUT_06_DATA_DIR / "06_candidate_features_for_modeling.csv",
    INPUT_06_TABLE_DIR / "06_candidate_features_for_modeling.csv",
], "06_candidate_features_for_modeling.csv")

PATH_ALL_RESULTS_DECISION = first_existing([
    INPUT_06_DATA_DIR / "06_significance_all_results_with_decision.csv",
    INPUT_06_TABLE_DIR / "06_significance_all_results_with_decision.csv",
], "06_significance_all_results_with_decision.csv", required=False)

PATH_06_FINAL_CHECKS = first_existing([
    INPUT_06_TABLE_DIR / "06_final_checks.csv",
    INPUT_06_TABLE_DIR / "06_final_output_check.csv",
], "06 final checks", required=False)

modeling = pd.read_csv(PATH_MODELING_TABLE)
candidate_features_raw = pd.read_csv(PATH_CANDIDATE_FEATURES)
all_results = pd.read_csv(PATH_ALL_RESULTS_DECISION) if PATH_ALL_RESULTS_DECISION is not None else pd.DataFrame()
final_checks_06 = pd.read_csv(PATH_06_FINAL_CHECKS) if PATH_06_FINAL_CHECKS is not None else pd.DataFrame()

input_file_summary = pd.DataFrame([
    {"name": "modeling_feature_table_with_content", "path": str(PATH_MODELING_TABLE), "rows": len(modeling), "cols": modeling.shape[1], "required": True},
    {"name": "06_candidate_features_for_modeling", "path": str(PATH_CANDIDATE_FEATURES), "rows": len(candidate_features_raw), "cols": candidate_features_raw.shape[1], "required": True},
    {"name": "06_significance_all_results_with_decision", "path": str(PATH_ALL_RESULTS_DECISION) if PATH_ALL_RESULTS_DECISION else None, "rows": len(all_results), "cols": all_results.shape[1] if len(all_results) else 0, "required": False},
    {"name": "06_final_checks", "path": str(PATH_06_FINAL_CHECKS) if PATH_06_FINAL_CHECKS else None, "rows": len(final_checks_06), "cols": final_checks_06.shape[1] if len(final_checks_06) else 0, "required": False},
])

input_file_summary.to_csv(OUTPUT_TABLE_DIR / "06_feature_redundancy_input_file_summary.csv", index=False, encoding="utf-8-sig")
display(input_file_summary)

required_model_cols = {ID_COL, TARGET_COL}
missing_required = sorted(required_model_cols - set(modeling.columns))
if missing_required:
    raise ValueError(f"modeling table 필수 컬럼 누락: {missing_required}")
if not modeling[ID_COL].is_unique:
    raise ValueError(f"{ID_COL}가 unique하지 않습니다. 07번 모델링 전 반드시 확인해야 합니다.")
if sorted(modeling[TARGET_COL].dropna().unique().tolist()) != [0, 1]:
    raise ValueError(f"{TARGET_COL}는 0/1 binary여야 합니다. 현재 값: {sorted(modeling[TARGET_COL].dropna().unique().tolist())}")

,name,path,rows,cols,required
0,modeling_feature_table_with_content,c:\Code\ott-churn-prediction\park.ingyeom\repo...,14922,230,True
1,06_candidate_features_for_modeling,c:\Code\ott-churn-prediction\park.ingyeom\repo...,39,33,True
2,06_significance_all_results_with_decision,c:\Code\ott-churn-prediction\park.ingyeom\repo...,1656,32,False
3,06_final_checks,c:\Code\ott-churn-prediction\park.ingyeom\repo...,32,3,False


## 5. 감사 대상 feature pool 구성

06번 후보 변수와 구조적으로 중복 위험이 큰 core 변수를 합쳐 감사합니다. 문자열 범주형 변수는 상관/VIF에서 원본 그대로 쓰지 않고 one-hot으로 처리하거나 별도 제외 사유를 남깁니다.

In [20]:
FEATURE_EXCLUDE = {ID_COL, TARGET_COL, "USER_KEY", "reg_date", "end_date", "product_code"}

if "feature" not in candidate_features_raw.columns:
    raise ValueError("06_candidate_features_for_modeling.csv에 feature 컬럼이 없습니다.")

candidate_feature_names = sorted(set(candidate_features_raw["feature"].dropna().astype(str)) - FEATURE_EXCLUDE)
force_features = [f for f in FORCE_AUDIT_FEATURES if f in modeling.columns and f not in FEATURE_EXCLUDE]
feature_pool = sorted(set(candidate_feature_names) | set(force_features))

feature_presence = pd.DataFrame({
    "feature": feature_pool,
    "exists_in_modeling": [f in modeling.columns for f in feature_pool],
    "from_06_candidate_features": [f in candidate_feature_names for f in feature_pool],
    "force_audit_feature": [f in force_features for f in feature_pool],
})
feature_presence.to_csv(OUTPUT_TABLE_DIR / "06_feature_redundancy_feature_presence.csv", index=False, encoding="utf-8-sig")

missing_in_modeling = feature_presence.loc[~feature_presence["exists_in_modeling"], "feature"].tolist()
if missing_in_modeling:
    print("modeling table에 없는 feature는 감사 대상에서 제외합니다:", missing_in_modeling)

feature_pool = [f for f in feature_pool if f in modeling.columns]

# 06 후보 파일에서 feature 단위 대표 메타데이터를 만듭니다.
def min_numeric(series: pd.Series) -> float:
    s = pd.to_numeric(series, errors="coerce")
    return float(s.min()) if s.notna().any() else np.nan

def max_abs_numeric(series: pd.Series) -> float:
    s = pd.to_numeric(series, errors="coerce").abs()
    return float(s.max()) if s.notna().any() else np.nan

def first_non_null(series: pd.Series):
    s = series.dropna()
    return s.iloc[0] if len(s) else np.nan

agg_spec = {
    "analysis_group": lambda s: ",".join(sorted(set(s.dropna().astype(str))))[:1000],
}
for col in ["feature_family", "feature_type", "decision", "source", "interpretation_note"]:
    if col in candidate_features_raw.columns:
        agg_spec[col] = first_non_null
for col in ["p_value", "p_value_fdr_global", "p_value_fdr_by_group", "p_value_fdr_by_group_family"]:
    if col in candidate_features_raw.columns:
        agg_spec[col] = min_numeric
for col in ["abs_diff", "effect_size", "rate_or_mean_diff"]:
    if col in candidate_features_raw.columns:
        agg_spec[col] = max_abs_numeric

candidate_meta = candidate_features_raw.groupby("feature", as_index=False).agg(agg_spec)

feature_inventory_rows = []
for f in feature_pool:
    s = modeling[f]
    non_null = int(s.notna().sum())
    n_unique = int(s.nunique(dropna=True))
    missing_rate = float(1 - non_null / len(modeling))
    is_numeric = bool(pd.api.types.is_numeric_dtype(s))
    is_binary_01 = False
    if is_numeric:
        vals = set(pd.to_numeric(s.dropna(), errors="coerce").dropna().unique().tolist())
        is_binary_01 = vals.issubset({0, 1}) and len(vals) <= 2
    feature_inventory_rows.append({
        "feature": f,
        "dtype": str(s.dtype),
        "is_numeric": is_numeric,
        "is_binary_0_1": is_binary_01,
        "non_null_n": non_null,
        "missing_rate": missing_rate,
        "n_unique": n_unique,
        "zero_variance": n_unique <= 1,
        "from_06_candidate_features": f in candidate_feature_names,
        "force_audit_feature": f in force_features,
    })

feature_inventory = pd.DataFrame(feature_inventory_rows)
feature_inventory = feature_inventory.merge(candidate_meta, on="feature", how="left")
feature_inventory.to_csv(OUTPUT_TABLE_DIR / "06_feature_redundancy_feature_inventory.csv", index=False, encoding="utf-8-sig")

feature_family_summary = (
    feature_inventory.groupby(["feature_family", "feature_type"], dropna=False)
    .agg(features=("feature", "count"), numeric_features=("is_numeric", "sum"), binary_features=("is_binary_0_1", "sum"))
    .reset_index()
    .sort_values(["features", "feature_family", "feature_type"], ascending=[False, True, True])
)
feature_family_summary.to_csv(OUTPUT_TABLE_DIR / "06_feature_redundancy_feature_family_summary.csv", index=False, encoding="utf-8-sig")

print("audit feature count:", len(feature_pool))
print("candidate feature count from 06:", len(candidate_feature_names))
print("force audit feature count found:", len(force_features))
if SHOW_TABLES:
    display(feature_inventory.head(50))
    display(feature_family_summary.head(30))

audit feature count: 54
candidate feature count from 06: 39
force audit feature count found: 28


## 6. 구조적 중복 및 완전 보완 관계 점검

상관계수 계산 이전에, 0/1 보완 관계나 완전 동일 변수처럼 구조적으로 중복되는 쌍을 먼저 점검합니다.

In [21]:
def numeric_series(feature: str) -> pd.Series:
    return pd.to_numeric(modeling[feature], errors="coerce")

def exact_equal_or_complement(s1: pd.Series, s2: pd.Series) -> tuple[bool, bool, int]:
    mask = s1.notna() & s2.notna()
    if mask.sum() == 0:
        return False, False, 0
    a = s1.loc[mask]
    b = s2.loc[mask]
    equal_flag = bool((a == b).all())
    complement_flag = False
    vals_a = set(a.unique().tolist())
    vals_b = set(b.unique().tolist())
    if vals_a.issubset({0, 1}) and vals_b.issubset({0, 1}):
        complement_flag = bool((a + b == 1).all())
    return equal_flag, complement_flag, int(mask.sum())

binary_features = feature_inventory.loc[feature_inventory["is_binary_0_1"], "feature"].tolist()
structural_rows = []
for i, f1 in enumerate(binary_features):
    s1 = numeric_series(f1)
    for f2 in binary_features[i+1:]:
        s2 = numeric_series(f2)
        equal_flag, complement_flag, n_pair = exact_equal_or_complement(s1, s2)
        if equal_flag or complement_flag:
            relation = "EXACT_DUPLICATE" if equal_flag else "EXACT_COMPLEMENT"
            structural_rows.append({
                "feature_a": f1,
                "feature_b": f2,
                "relation": relation,
                "non_null_pair_n": n_pair,
                "recommendation": "선형 모델 또는 SHAP 해석에서는 둘 중 하나만 유지하는 것을 우선 검토",
            })

structural_duplicates = pd.DataFrame(structural_rows).sort_values(["relation", "feature_a", "feature_b"]) if structural_rows else pd.DataFrame(columns=[
    "feature_a", "feature_b", "relation", "non_null_pair_n", "recommendation"
])
structural_duplicates.to_csv(OUTPUT_TABLE_DIR / "06_feature_redundancy_structural_duplicates.csv", index=False, encoding="utf-8-sig")

print("structural duplicate/complement pairs:", len(structural_duplicates))
if SHOW_TABLES:
    display(structural_duplicates.head(50))

structural duplicate/complement pairs: 17


## 7. Spearman 고상관 변수쌍 점검

수치형 변수와 이진형 변수에 대해 pairwise 상관관계를 계산하고, 임계치 이상인 변수쌍을 저장합니다.

In [22]:
corr_features = feature_inventory.loc[
    (feature_inventory["is_numeric"]) &
    (~feature_inventory["zero_variance"]) &
    (feature_inventory["non_null_n"] >= MIN_CORR_NON_NULL_N) &
    (feature_inventory["missing_rate"] <= (1 - MIN_NON_NULL_RATIO)),
    "feature"
].tolist()

corr_input = modeling[corr_features].apply(pd.to_numeric, errors="coerce")
corr_matrix = corr_input.corr(method=CORR_METHOD, min_periods=MIN_CORR_NON_NULL_N)

pairs = []
for i, f1 in enumerate(corr_features):
    for f2 in corr_features[i+1:]:
        corr_val = corr_matrix.loc[f1, f2]
        if pd.isna(corr_val):
            continue
        abs_corr = abs(float(corr_val))
        if abs_corr >= HIGH_CORR_THRESHOLD:
            valid_n = int((corr_input[f1].notna() & corr_input[f2].notna()).sum())
            severity = "NEAR_DUPLICATE" if abs_corr >= NEAR_DUPLICATE_CORR_THRESHOLD else "HIGH_CORR"
            direction = "positive" if corr_val >= 0 else "negative"
            pairs.append({
                "feature_a": f1,
                "feature_b": f2,
                "corr_method": CORR_METHOD,
                "corr": float(corr_val),
                "abs_corr": abs_corr,
                "corr_direction": direction,
                "non_null_pair_n": valid_n,
                "severity": severity,
                "recommendation": "07번 모델링에서 둘 다 넣는 경우 계수/SHAP 해석 주의",
            })

high_corr_pairs = pd.DataFrame(pairs).sort_values(["abs_corr", "non_null_pair_n"], ascending=[False, False]) if pairs else pd.DataFrame(columns=[
    "feature_a", "feature_b", "corr_method", "corr", "abs_corr", "corr_direction", "non_null_pair_n", "severity", "recommendation"
])

meta_cols = [c for c in ["feature", "feature_family", "feature_type", "decision", "p_value_fdr_by_group_family", "abs_diff", "effect_size"] if c in feature_inventory.columns]
meta = feature_inventory[meta_cols].copy()
high_corr_pairs = high_corr_pairs.merge(meta.add_prefix("a_"), left_on="feature_a", right_on="a_feature", how="left").drop(columns=["a_feature"], errors="ignore")
high_corr_pairs = high_corr_pairs.merge(meta.add_prefix("b_"), left_on="feature_b", right_on="b_feature", how="left").drop(columns=["b_feature"], errors="ignore")

corr_matrix_summary = corr_matrix.reset_index().rename(columns={"index": "feature"})
corr_matrix_summary.to_csv(OUTPUT_DATA_DIR / "06_feature_redundancy_corr_matrix.csv", index=False, encoding="utf-8-sig")
high_corr_pairs.to_csv(OUTPUT_TABLE_DIR / "06_feature_redundancy_high_corr_pairs.csv", index=False, encoding="utf-8-sig")

print("correlation audit features:", len(corr_features))
print("high correlation pairs:", len(high_corr_pairs))
if SHOW_TABLES:
    display(high_corr_pairs.head(50))

correlation audit features: 51
high correlation pairs: 46


## 8. 고상관 클러스터 구성

고상관 pair를 그래프의 edge로 보고 connected component를 만들어 redundancy cluster를 구성합니다.

In [23]:
def build_components(edges: pd.DataFrame) -> list[list[str]]:
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    nodes = set(edges["feature_a"].tolist()) | set(edges["feature_b"].tolist())
    for n in nodes:
        parent.setdefault(n, n)
    for _, row in edges.iterrows():
        union(row["feature_a"], row["feature_b"])

    comps = {}
    for n in nodes:
        root = find(n)
        comps.setdefault(root, []).append(n)
    return [sorted(v) for v in comps.values()]

components = build_components(high_corr_pairs[["feature_a", "feature_b"]]) if len(high_corr_pairs) else []
cluster_rows = []
for cid, feats in enumerate(sorted(components, key=lambda x: (-len(x), x)), start=1):
    cluster_candidate_decisions = feature_inventory.loc[feature_inventory["feature"].isin(feats), "decision"].dropna().astype(str).tolist()
    representative = None
    if cluster_candidate_decisions:
        decision_priority = ["KEEP", "REVIEW", "DROP_CANDIDATE", "DROP"]
        tmp = feature_inventory.loc[feature_inventory["feature"].isin(feats), ["feature", "decision", "missing_rate"]].copy()
        tmp["decision_rank"] = tmp["decision"].apply(lambda x: decision_priority.index(x) if x in decision_priority else 99)
        tmp = tmp.sort_values(["decision_rank", "missing_rate", "feature"], ascending=[True, True, True])
        representative = tmp.iloc[0]["feature"]
    else:
        tmp = feature_inventory.loc[feature_inventory["feature"].isin(feats), ["feature", "missing_rate"]].copy()
        tmp = tmp.sort_values(["missing_rate", "feature"], ascending=[True, True])
        representative = tmp.iloc[0]["feature"]

    for f in feats:
        cluster_rows.append({
            "cluster_id": cid,
            "cluster_size": len(feats),
            "cluster_representative": representative,
            "feature": f,
            "is_representative": f == representative,
        })

redundancy_clusters = pd.DataFrame(cluster_rows).sort_values(["cluster_size", "cluster_id", "is_representative", "feature"], ascending=[False, True, False, True]) if cluster_rows else pd.DataFrame(columns=[
    "cluster_id", "cluster_size", "cluster_representative", "feature", "is_representative"
])
redundancy_clusters.to_csv(OUTPUT_TABLE_DIR / "06_feature_redundancy_clusters.csv", index=False, encoding="utf-8-sig")

cluster_summary = redundancy_clusters.groupby("cluster_id", as_index=False).agg(
    cluster_size=("cluster_size", "max"),
    cluster_representative=("cluster_representative", "first"),
    features=("feature", lambda s: ", ".join(s.astype(str)))
).sort_values(["cluster_size", "cluster_id"], ascending=[False, True]) if len(redundancy_clusters) else pd.DataFrame(columns=["cluster_id", "cluster_size", "cluster_representative", "features"])
cluster_summary.to_csv(OUTPUT_TABLE_DIR / "06_feature_redundancy_cluster_summary.csv", index=False, encoding="utf-8-sig")

print("redundancy clusters:", redundancy_clusters["cluster_id"].nunique() if len(redundancy_clusters) else 0)
if SHOW_TABLES:
    display(cluster_summary.head(30))

redundancy clusters: 11


## 9. VIF 계산

VIF는 주로 선형/로지스틱 회귀 계수 해석 안정성을 점검하기 위한 지표입니다. 이 노트북에서는 07번 모델링 전 “주의가 필요한 변수”를 표시하는 용도로 사용합니다.

In [24]:
def infer_categorical_features(features: list[str]) -> list[str]:
    cats = []
    for f in features:
        s = modeling[f]
        if not pd.api.types.is_numeric_dtype(s):
            cats.append(f)
            continue
    return cats

def make_vif_matrix(features: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    usable_features = []
    for f in features:
        s = modeling[f]
        missing_rate = float(s.isna().mean())
        n_unique = int(s.nunique(dropna=True))
        if n_unique <= 1:
            rows.append({"feature": f, "include_for_vif": False, "reason": "zero_variance"})
            continue
        if missing_rate > MAX_MISSING_RATIO_FOR_VIF:
            rows.append({"feature": f, "include_for_vif": False, "reason": f"missing_rate>{MAX_MISSING_RATIO_FOR_VIF}"})
            continue
        usable_features.append(f)
        rows.append({"feature": f, "include_for_vif": True, "reason": "included"})

    categorical = infer_categorical_features(usable_features)
    numeric = [f for f in usable_features if f not in categorical]

    pieces = []
    encoded_map = []
    for f in numeric:
        x = pd.to_numeric(modeling[f], errors="coerce")
        fill = x.median() if x.notna().any() else 0
        x = x.fillna(fill).astype(float)
        pieces.append(pd.DataFrame({f: x}))
        encoded_map.append({"encoded_feature": f, "original_feature": f, "encoding": "numeric", "include_for_vif": True})

    for f in categorical:
        cat = modeling[f].astype("object").where(modeling[f].notna(), "__MISSING__")
        dummies = pd.get_dummies(cat, prefix=f, drop_first=True, dtype=float)
        if dummies.shape[1] == 0:
            continue
        pieces.append(dummies)
        for col in dummies.columns:
            encoded_map.append({"encoded_feature": col, "original_feature": f, "encoding": "one_hot_drop_first", "include_for_vif": True})

    if pieces:
        X = pd.concat(pieces, axis=1)
        X = X.loc[:, X.nunique(dropna=True) > 1]
    else:
        X = pd.DataFrame(index=modeling.index)

    vif_feature_status = pd.DataFrame(rows)
    encoded_map_df = pd.DataFrame(encoded_map)
    if len(encoded_map_df):
        encoded_map_df = encoded_map_df[encoded_map_df["encoded_feature"].isin(X.columns)]
    return X, vif_feature_status.merge(encoded_map_df, left_on="feature", right_on="original_feature", how="left")

def compute_vif_numpy(X: pd.DataFrame) -> pd.DataFrame:
    if X.shape[1] == 0:
        return pd.DataFrame(columns=["encoded_feature", "vif", "r2_approx", "severity", "corr_matrix_rank", "corr_matrix_cols", "rank_deficient"])

    X_num = X.astype(float).to_numpy()
    means = np.nanmean(X_num, axis=0)
    stds = np.nanstd(X_num, axis=0)
    stds[stds == 0] = 1.0
    X_std = (X_num - means) / stds
    X_std = np.nan_to_num(X_std, nan=0.0, posinf=0.0, neginf=0.0)

    corr = np.corrcoef(X_std, rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    np.fill_diagonal(corr, 1.0)

    p = corr.shape[0]
    rank = int(np.linalg.matrix_rank(corr, tol=1e-10))
    rank_deficient = rank < p
    inv_corr = np.linalg.pinv(corr) if rank_deficient else np.linalg.inv(corr)
    vif_values = np.diag(inv_corr).astype(float)
    vif_values = np.where(vif_values < 1, 1.0, vif_values)
    r2_approx = 1.0 - (1.0 / vif_values)

    rows = []
    for col, vif, r2 in zip(X.columns, vif_values, r2_approx):
        if np.isinf(vif) or vif >= VIF_HIGH_THRESHOLD:
            severity = "HIGH"
        elif vif >= VIF_WARN_THRESHOLD:
            severity = "WARN"
        else:
            severity = "OK"
        rows.append({
            "encoded_feature": col,
            "vif": float(vif),
            "r2_approx": float(r2),
            "severity": severity,
            "corr_matrix_rank": rank,
            "corr_matrix_cols": p,
            "rank_deficient": rank_deficient,
        })
    return pd.DataFrame(rows).sort_values(["severity", "vif"], ascending=[True, False])

vif_candidate_features = [f for f in feature_pool if f in modeling.columns]
X_vif, vif_feature_map = make_vif_matrix(vif_candidate_features)

if RUN_VIF:
    vif_encoded = compute_vif_numpy(X_vif)
else:
    vif_encoded = pd.DataFrame(columns=["encoded_feature", "vif", "r2_approx", "severity"])

vif_candidates = vif_encoded.merge(vif_feature_map, on="encoded_feature", how="left") if len(vif_encoded) else vif_encoded.copy()
if len(vif_candidates):
    vif_candidates = vif_candidates.merge(
        feature_inventory[["feature", "feature_family", "feature_type", "decision"]].rename(columns={"feature": "original_feature"}),
        on="original_feature",
        how="left",
    )

vif_feature_map.to_csv(OUTPUT_DATA_DIR / "06_feature_redundancy_vif_feature_map.csv", index=False, encoding="utf-8-sig")
vif_candidates.to_csv(OUTPUT_TABLE_DIR / "06_feature_vif_candidates.csv", index=False, encoding="utf-8-sig")

print("VIF matrix shape:", X_vif.shape)
print("VIF rows:", len(vif_candidates))
print("VIF HIGH/WARN counts:", vif_candidates["severity"].value_counts(dropna=False).to_dict() if len(vif_candidates) else {})
if SHOW_TABLES:
    display(vif_candidates.head(50))

VIF matrix shape: (14922, 63)
VIF rows: 63
VIF HIGH/WARN counts: {'OK': 47, 'HIGH': 9, 'WARN': 7}


## 10. 다중공선성 관련 변수쌍 설명표 생성

VIF가 높다고 해서 어떤 변수와의 중복에서 비롯되었는지가 바로 보이지는 않습니다. 그래서 VIF가 높은 변수를 기준으로, 구조적 duplicate/complement와 high-corr pair를 함께 묶어 “어떤 변수와 어떤 변수 간 중복이 의심되는지”를 보여주는 표를 만듭니다.

In [25]:
# 양방향 구조적 관계 테이블
structural_long_rows = []
for _, row in structural_duplicates.iterrows():
    structural_long_rows.append({
        "focus_feature": row["feature_a"],
        "related_feature": row["feature_b"],
        "relation_type": row["relation"],
        "relation_strength": 1.0,
        "relation_detail": row["relation"],
        "evidence_source": "structural_duplicates",
    })
    structural_long_rows.append({
        "focus_feature": row["feature_b"],
        "related_feature": row["feature_a"],
        "relation_type": row["relation"],
        "relation_strength": 1.0,
        "relation_detail": row["relation"],
        "evidence_source": "structural_duplicates",
    })
structural_long = pd.DataFrame(structural_long_rows)

# 양방향 고상관 관계 테이블
high_corr_long_rows = []
for _, row in high_corr_pairs.iterrows():
    high_corr_long_rows.append({
        "focus_feature": row["feature_a"],
        "related_feature": row["feature_b"],
        "relation_type": "HIGH_CORR_PAIR",
        "relation_strength": float(row["abs_corr"]),
        "relation_detail": f"corr={row['corr']:.4f}",
        "evidence_source": "high_corr_pairs",
    })
    high_corr_long_rows.append({
        "focus_feature": row["feature_b"],
        "related_feature": row["feature_a"],
        "relation_type": "HIGH_CORR_PAIR",
        "relation_strength": float(row["abs_corr"]),
        "relation_detail": f"corr={row['corr']:.4f}",
        "evidence_source": "high_corr_pairs",
    })
high_corr_long = pd.DataFrame(high_corr_long_rows)

all_related = pd.concat([structural_long, high_corr_long], ignore_index=True) if len(structural_long) or len(high_corr_long) else pd.DataFrame(columns=[
    "focus_feature", "related_feature", "relation_type", "relation_strength", "relation_detail", "evidence_source"
])

vif_focus = pd.DataFrame(columns=["original_feature", "max_vif", "max_vif_severity"])
if len(vif_candidates):
    vif_focus = (
        vif_candidates.groupby("original_feature", as_index=False)
        .agg(max_vif=("vif", "max"), max_vif_severity=("severity", lambda s: "HIGH" if "HIGH" in set(s.astype(str)) else ("WARN" if "WARN" in set(s.astype(str)) else "OK")))
        .sort_values(["max_vif", "original_feature"], ascending=[False, True])
    )
    vif_focus = vif_focus.loc[vif_focus["max_vif_severity"].isin(["HIGH", "WARN"])]

related_rows = []
for _, row in vif_focus.iterrows():
    feature = row["original_feature"]
    sub = all_related.loc[all_related["focus_feature"] == feature].copy()
    if len(sub) == 0:
        related_rows.append({
            "focus_feature": feature,
            "max_vif": row["max_vif"],
            "max_vif_severity": row["max_vif_severity"],
            "related_feature": np.nan,
            "relation_type": "NO_DIRECT_PAIR_FOUND",
            "relation_strength": np.nan,
            "relation_detail": "VIF는 높지만 HIGH_CORR_THRESHOLD 이상 pair는 확인되지 않음",
            "evidence_source": "vif_only",
            "rank_within_feature": 1,
        })
        continue

    sub = sub.sort_values(["relation_strength", "related_feature"], ascending=[False, True]).drop_duplicates(["focus_feature", "related_feature", "relation_type"])
    sub = sub.head(MAX_RELATED_PER_FEATURE)
    for rank, (_, srow) in enumerate(sub.iterrows(), start=1):
        related_rows.append({
            "focus_feature": feature,
            "max_vif": row["max_vif"],
            "max_vif_severity": row["max_vif_severity"],
            "related_feature": srow["related_feature"],
            "relation_type": srow["relation_type"],
            "relation_strength": srow["relation_strength"],
            "relation_detail": srow["relation_detail"],
            "evidence_source": srow["evidence_source"],
            "rank_within_feature": rank,
        })

multicollinearity_related = pd.DataFrame(related_rows)
multicollinearity_related.to_csv(OUTPUT_TABLE_DIR / "06_feature_multicollinearity_related_features.csv", index=False, encoding="utf-8-sig")

multicollinearity_summary = (
    multicollinearity_related.groupby(["focus_feature", "max_vif", "max_vif_severity"], dropna=False)
    .agg(
        related_feature_count=("related_feature", lambda s: int(s.notna().sum())),
        related_features=("related_feature", lambda s: ", ".join(pd.Series(s.dropna().astype(str).unique()).head(MAX_RELATED_PER_FEATURE))),
        relation_types=("relation_type", lambda s: ", ".join(pd.Series(s.astype(str).unique()))),
    )
    .reset_index()
    .sort_values(["max_vif", "focus_feature"], ascending=[False, True])
) if len(multicollinearity_related) else pd.DataFrame(columns=["focus_feature", "max_vif", "max_vif_severity", "related_feature_count", "related_features", "relation_types"])
multicollinearity_summary.to_csv(OUTPUT_TABLE_DIR / "06_feature_multicollinearity_summary.csv", index=False, encoding="utf-8-sig")

print("multicollinearity-related rows:", len(multicollinearity_related))
if SHOW_TABLES:
    display(multicollinearity_related.head(50))
    display(multicollinearity_summary.head(30))

multicollinearity-related rows: 19


## 11. 07번 모델링 후보 변수 검토표 생성

각 변수에 대해 고상관 연결 수, 최대 상관계수, VIF, 클러스터 대표 여부를 종합하여 KEEP / REVIEW / DROP_CANDIDATE 참고용 감사지표를 만듭니다.

In [26]:
high_corr_degree = pd.Series(dtype=int)
max_abs_corr = pd.Series(dtype=float)
if len(high_corr_pairs):
    a_side = high_corr_pairs.groupby("feature_a").agg(high_corr_pair_count=("feature_b", "count"), max_abs_corr=("abs_corr", "max")).rename_axis("feature")
    b_side = high_corr_pairs.groupby("feature_b").agg(high_corr_pair_count=("feature_a", "count"), max_abs_corr=("abs_corr", "max")).rename_axis("feature")
    merged = pd.concat([a_side, b_side], axis=0).groupby(level=0).agg({"high_corr_pair_count": "sum", "max_abs_corr": "max"})
    high_corr_degree = merged["high_corr_pair_count"]
    max_abs_corr = merged["max_abs_corr"]

cluster_map = redundancy_clusters[["feature", "cluster_id", "cluster_representative", "cluster_size", "is_representative"]].copy() if len(redundancy_clusters) else pd.DataFrame(columns=["feature", "cluster_id", "cluster_representative", "cluster_size", "is_representative"])

vif_by_feature = pd.DataFrame(columns=["feature", "max_vif", "max_vif_severity"])
if len(vif_candidates):
    vif_tmp = vif_candidates.groupby("original_feature", as_index=False).agg(
        max_vif=("vif", "max"),
        max_vif_severity=("severity", lambda s: "HIGH" if "HIGH" in set(s.astype(str)) else ("WARN" if "WARN" in set(s.astype(str)) else "OK"))
    ).rename(columns={"original_feature": "feature"})
    vif_by_feature = vif_tmp

candidate_review = feature_inventory.copy()
candidate_review["high_corr_pair_count"] = candidate_review["feature"].map(high_corr_degree).fillna(0).astype(int)
candidate_review["max_abs_corr"] = candidate_review["feature"].map(max_abs_corr)
candidate_review = candidate_review.merge(cluster_map, on="feature", how="left")
candidate_review = candidate_review.merge(vif_by_feature, on="feature", how="left")

# 구조적 duplicate/complement 개수
if len(structural_duplicates):
    structural_degree = pd.concat([
        structural_duplicates.groupby("feature_a").size().rename_axis("feature"),
        structural_duplicates.groupby("feature_b").size().rename_axis("feature"),
    ]).groupby(level=0).sum()
else:
    structural_degree = pd.Series(dtype=int)
candidate_review["structural_relation_count"] = candidate_review["feature"].map(structural_degree).fillna(0).astype(int)

# 제안 액션 규칙
proposed_action = []
audit_note = []
for _, row in candidate_review.iterrows():
    note_parts = []
    if row["structural_relation_count"] > 0:
        note_parts.append("구조적 duplicate/complement 관계 존재")
    if row["high_corr_pair_count"] >= 2:
        note_parts.append("고상관 연결이 많음")
    if pd.notna(row.get("max_vif")) and row.get("max_vif", 0) >= VIF_HIGH_THRESHOLD:
        note_parts.append("VIF HIGH")
    elif pd.notna(row.get("max_vif")) and row.get("max_vif", 0) >= VIF_WARN_THRESHOLD:
        note_parts.append("VIF WARN")
    if pd.notna(row.get("cluster_id")) and not bool(row.get("is_representative", False)):
        note_parts.append(f"cluster 대표 변수는 {row.get('cluster_representative')}")

    if row["structural_relation_count"] > 0:
        action = "DROP_CANDIDATE"
    elif pd.notna(row.get("cluster_id")) and not bool(row.get("is_representative", False)) and row["high_corr_pair_count"] > 0:
        action = "REVIEW"
    elif pd.notna(row.get("max_vif")) and row.get("max_vif", 0) >= VIF_HIGH_THRESHOLD:
        action = "REVIEW"
    else:
        action = "KEEP"

    proposed_action.append(action)
    audit_note.append("; ".join(note_parts) if note_parts else "특이 중복 신호 없음")

candidate_review["proposed_action"] = proposed_action
candidate_review["audit_note"] = audit_note
candidate_review = candidate_review.sort_values([
    "proposed_action", "high_corr_pair_count", "max_abs_corr", "max_vif", "missing_rate", "feature"
], ascending=[True, False, False, False, True, True])

candidate_review.to_csv(OUTPUT_DATA_DIR / "06_feature_modeling_candidate_redundancy_review.csv", index=False, encoding="utf-8-sig")

refined_candidates = candidate_review.loc[
    candidate_review["from_06_candidate_features"],
    ["feature", "decision", "feature_family", "feature_type", "proposed_action", "high_corr_pair_count", "structural_relation_count", "max_abs_corr", "max_vif", "max_vif_severity", "cluster_representative", "audit_note"]
].copy()
refined_candidates.to_csv(OUTPUT_DATA_DIR / "06_feature_modeling_candidate_refined.csv", index=False, encoding="utf-8-sig")

print("candidate review rows:", len(candidate_review))
print("refined 06 candidate rows:", len(refined_candidates))
print("proposed action counts:", candidate_review["proposed_action"].value_counts(dropna=False).to_dict())
if SHOW_TABLES:
    display(refined_candidates.head(80))

candidate review rows: 54
refined 06 candidate rows: 39
proposed action counts: {'KEEP': 25, 'REVIEW': 19, 'DROP_CANDIDATE': 10}


## 12. 요약표 및 시각화

여기서는 감사지표를 사람이 빠르게 읽을 수 있도록 요약표와 시각화를 만듭니다. 특히 “어떤 변수와 어떤 변수 사이의 다중공선성이 의심되는가”가 보이도록 관련 변수표와 heatmap을 남깁니다.

In [27]:
high_corr_summary = pd.DataFrame([
    {"metric": "corr_method", "value": CORR_METHOD},
    {"metric": "corr_features", "value": len(corr_features)},
    {"metric": "high_corr_threshold", "value": HIGH_CORR_THRESHOLD},
    {"metric": "high_corr_pairs", "value": len(high_corr_pairs)},
    {"metric": "near_duplicate_corr_threshold", "value": NEAR_DUPLICATE_CORR_THRESHOLD},
    {"metric": "near_duplicate_pairs", "value": int((high_corr_pairs["severity"] == "NEAR_DUPLICATE").sum()) if len(high_corr_pairs) else 0},
    {"metric": "redundancy_cluster_count", "value": redundancy_clusters["cluster_id"].nunique() if len(redundancy_clusters) else 0},
])
high_corr_summary.to_csv(OUTPUT_TABLE_DIR / "06_feature_redundancy_high_corr_summary.csv", index=False, encoding="utf-8-sig")

vif_summary = pd.DataFrame([
    {"metric": "run_vif", "value": RUN_VIF},
    {"metric": "vif_matrix_rows", "value": X_vif.shape[0]},
    {"metric": "vif_matrix_cols", "value": X_vif.shape[1]},
    {"metric": "vif_warn_threshold", "value": VIF_WARN_THRESHOLD},
    {"metric": "vif_high_threshold", "value": VIF_HIGH_THRESHOLD},
    {"metric": "vif_warn_or_high_rows", "value": int(vif_candidates["severity"].isin(["WARN", "HIGH"]).sum()) if len(vif_candidates) else 0},
    {"metric": "vif_high_rows", "value": int((vif_candidates["severity"] == "HIGH").sum()) if len(vif_candidates) else 0},
])
vif_summary.to_csv(OUTPUT_TABLE_DIR / "06_feature_redundancy_vif_summary.csv", index=False, encoding="utf-8-sig")

action_summary = candidate_review["proposed_action"].value_counts(dropna=False).rename_axis("proposed_action").reset_index(name="feature_count")
action_summary.to_csv(OUTPUT_TABLE_DIR / "06_feature_redundancy_action_summary.csv", index=False, encoding="utf-8-sig")

decision_notes = pd.DataFrame([
    {"topic": "VIF 해석", "note": "VIF는 주로 선형/로지스틱 회귀 계수 해석의 안정성 진단이다. 트리 모델 성능 조건은 아니지만 SHAP 해석에서는 중복 변수가 중요도를 나눌 수 있다."},
    {"topic": "고상관 변수쌍", "note": f"abs(Spearman corr) >= {HIGH_CORR_THRESHOLD}인 변수쌍은 07번 모델링에서 동시에 투입할지 검토한다."},
    {"topic": "구조적 보완 관계", "note": "0/1 보완 관계나 완전 동일 변수는 선형 모델과 SHAP 해석에서 둘 중 하나만 쓰는 것을 우선 검토한다."},
    {"topic": "자동 삭제 금지", "note": "이 노트북은 변수를 자동 제거하지 않는다. 07번 모델링 전 후보 변수 조합을 검토하기 위한 감사표를 만든다."},
    {"topic": "관련 변수표", "note": "06_feature_multicollinearity_related_features.csv에서 VIF가 높은 변수와 직접 연결된 관련 변수들을 확인할 수 있다."},
])
decision_notes.to_csv(OUTPUT_TABLE_DIR / "06_feature_redundancy_decision_notes.csv", index=False, encoding="utf-8-sig")

summary_json = {
    "notebook_id": NOTEBOOK_ID,
    "input_rows": int(len(modeling)),
    "input_columns": int(modeling.shape[1]),
    "candidate_features_from_06": int(len(candidate_feature_names)),
    "audit_feature_count": int(len(feature_pool)),
    "high_corr_pairs": int(len(high_corr_pairs)),
    "structural_duplicate_or_complement_pairs": int(len(structural_duplicates)),
    "redundancy_cluster_count": int(redundancy_clusters["cluster_id"].nunique() if len(redundancy_clusters) else 0),
    "vif_rows": int(len(vif_candidates)),
    "vif_high_rows": int((vif_candidates["severity"] == "HIGH").sum()) if len(vif_candidates) else 0,
    "vif_warn_or_high_rows": int(vif_candidates["severity"].isin(["WARN", "HIGH"]).sum()) if len(vif_candidates) else 0,
    "multicollinearity_related_rows": int(len(multicollinearity_related)),
}
(OUTPUT_DATA_DIR / "06_feature_redundancy_summary.json").write_text(json.dumps(summary_json, ensure_ascii=False, indent=2), encoding="utf-8")

# ---- Figure helper ----
def save_current_fig(filename: str):
    path = OUTPUT_FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()
    return path

# 1) 고상관 pair bar chart
if RUN_FIGURES and len(high_corr_pairs):
    top_pairs = high_corr_pairs.head(30).copy()
    labels = top_pairs["feature_a"] + " ↔ " + top_pairs["feature_b"]
    fig_h = max(6, 0.32 * len(top_pairs))
    plt.figure(figsize=(13, fig_h))
    plt.barh(labels[::-1], top_pairs["abs_corr"].values[::-1])
    plt.xlabel(f"|{CORR_METHOD} 상관계수|")
    plt.title("상위 고상관 변수쌍")
    save_current_fig("06_feature_redundancy_top_high_corr_pairs.png")

# 2) VIF bar chart
if RUN_FIGURES and len(vif_candidates):
    top_vif = vif_candidates.copy()
    top_vif["vif_plot"] = top_vif["vif"].replace(np.inf, np.nan)
    finite = top_vif.dropna(subset=["vif_plot"]).sort_values("vif_plot", ascending=False).head(30)
    if len(finite):
        labels = finite["encoded_feature"]
        fig_h = max(6, 0.32 * len(finite))
        plt.figure(figsize=(13, fig_h))
        plt.barh(labels[::-1], finite["vif_plot"].values[::-1])
        plt.axvline(VIF_WARN_THRESHOLD, linestyle="--", linewidth=1)
        plt.axvline(VIF_HIGH_THRESHOLD, linestyle="--", linewidth=1)
        plt.xlabel("VIF")
        plt.title("상위 VIF 변수")
        save_current_fig("06_feature_redundancy_top_vif.png")

# 3) 고상관 heatmap
if RUN_FIGURES and len(high_corr_pairs):
    degree_order = pd.concat([
        high_corr_pairs[["feature_a", "abs_corr"]].rename(columns={"feature_a": "feature"}),
        high_corr_pairs[["feature_b", "abs_corr"]].rename(columns={"feature_b": "feature"}),
    ]).groupby("feature").agg(pair_count=("abs_corr", "count"), max_abs_corr=("abs_corr", "max")).sort_values(["pair_count", "max_abs_corr", "feature"], ascending=[False, False, True])
    heatmap_features = [f for f in degree_order.head(MAX_HEATMAP_FEATURES).index.tolist() if f in corr_matrix.index]
    if len(heatmap_features) >= 2:
        hm = corr_matrix.loc[heatmap_features, heatmap_features]
        plt.figure(figsize=(12, 10))
        plt.imshow(hm, cmap="coolwarm", vmin=-1, vmax=1)
        plt.xticks(range(len(heatmap_features)), heatmap_features, rotation=90)
        plt.yticks(range(len(heatmap_features)), heatmap_features)
        plt.colorbar(label="상관계수")
        plt.title("고상관 변수 중심 상관행렬 heatmap")
        save_current_fig("06_feature_redundancy_top_corr_heatmap.png")

# 4) 클러스터 크기 bar chart
if RUN_FIGURES and len(cluster_summary):
    top_clusters = cluster_summary.head(20).copy()
    labels = top_clusters["cluster_id"].astype(str) + " | 대표: " + top_clusters["cluster_representative"].astype(str)
    fig_h = max(5, 0.35 * len(top_clusters))
    plt.figure(figsize=(13, fig_h))
    plt.barh(labels[::-1], top_clusters["cluster_size"].values[::-1])
    plt.xlabel("클러스터 내 변수 수")
    plt.title("고상관 redundancy cluster 크기")
    save_current_fig("06_feature_redundancy_cluster_sizes.png")

# 5) VIF 높은 변수와 관련 변수 heatmap
if RUN_FIGURES and len(multicollinearity_related):
    focus_features = multicollinearity_summary.head(10)["focus_feature"].dropna().astype(str).tolist()
    related_features = multicollinearity_related.loc[multicollinearity_related["focus_feature"].isin(focus_features), "related_feature"].dropna().astype(str).tolist()
    heatmap_features = []
    for f in focus_features + related_features:
        if f in corr_matrix.index and f not in heatmap_features:
            heatmap_features.append(f)
        if len(heatmap_features) >= MAX_HEATMAP_FEATURES:
            break
    if len(heatmap_features) >= 2:
        hm = corr_matrix.loc[heatmap_features, heatmap_features]
        plt.figure(figsize=(12, 10))
        plt.imshow(hm, cmap="coolwarm", vmin=-1, vmax=1)
        plt.xticks(range(len(heatmap_features)), heatmap_features, rotation=90)
        plt.yticks(range(len(heatmap_features)), heatmap_features)
        plt.colorbar(label="상관계수")
        plt.title("VIF 높은 변수와 관련 변수 heatmap")
        save_current_fig("06_feature_redundancy_high_vif_related_heatmap.png")

# 6) 액션 분포 chart
if RUN_FIGURES and len(action_summary):
    plt.figure(figsize=(8, 5))
    plt.bar(action_summary["proposed_action"].astype(str), action_summary["feature_count"].astype(float))
    plt.xlabel("제안 액션")
    plt.ylabel("변수 수")
    plt.title("후보 변수 감사 결과 액션 분포")
    save_current_fig("06_feature_redundancy_action_counts.png")

# 7) 상위 pair detail scatter figure
if RUN_FIGURES and len(high_corr_pairs):
    top_detail = high_corr_pairs.head(TOP_PAIR_DETAIL_COUNT).copy()
    if len(top_detail):
        n = len(top_detail)
        cols = 2
        rows = int(math.ceil(n / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(14, max(4 * rows, 5)))
        axes = np.array(axes).reshape(-1)
        rng = np.random.default_rng(42)
        for ax, (_, prow) in zip(axes, top_detail.iterrows()):
            x = pd.to_numeric(modeling[prow["feature_a"]], errors="coerce")
            y = pd.to_numeric(modeling[prow["feature_b"]], errors="coerce")
            mask = x.notna() & y.notna()
            x_plot = x.loc[mask].astype(float).to_numpy()
            y_plot = y.loc[mask].astype(float).to_numpy()
            # binary/binary 겹침 완화용 jitter
            if len(x_plot):
                if set(np.unique(x_plot)).issubset({0.0, 1.0}):
                    x_plot = x_plot + rng.normal(0, 0.02, size=len(x_plot))
                if set(np.unique(y_plot)).issubset({0.0, 1.0}):
                    y_plot = y_plot + rng.normal(0, 0.02, size=len(y_plot))
            ax.scatter(x_plot, y_plot, alpha=0.25, s=8)
            ax.set_xlabel(prow["feature_a"])
            ax.set_ylabel(prow["feature_b"])
            ax.set_title(f"corr={prow['corr']:.3f} | {prow['severity']}")
        for ax in axes[n:]:
            ax.axis("off")
        fig.suptitle("상위 고상관 변수쌍 상세 분포", y=1.02)
        plt.tight_layout()
        plt.savefig(OUTPUT_FIGURE_DIR / "06_feature_redundancy_top_pair_details.png", dpi=FIG_DPI, bbox_inches="tight")
        plt.close(fig)

print(json.dumps(summary_json, ensure_ascii=False, indent=2))

{
  "notebook_id": "06_feature_redundancy_audit",
  "input_rows": 14922,
  "input_columns": 230,
  "candidate_features_from_06": 39,
  "audit_feature_count": 54,
  "high_corr_pairs": 46,
  "structural_duplicate_or_complement_pairs": 17,
  "redundancy_cluster_count": 11,
  "vif_rows": 63,
  "vif_high_rows": 9,
  "vif_warn_or_high_rows": 16,
  "multicollinearity_related_rows": 19
}


## 13. 최종 검산

이 노트북의 최종 검산은 “고상관/구조적 duplicate/VIF/관련 변수표가 모두 생성되었는가”를 확인합니다.

In [28]:
expected_data_files = [
    OUTPUT_DATA_DIR / "06_feature_redundancy_corr_matrix.csv",
    OUTPUT_DATA_DIR / "06_feature_redundancy_vif_feature_map.csv",
    OUTPUT_DATA_DIR / "06_feature_modeling_candidate_redundancy_review.csv",
    OUTPUT_DATA_DIR / "06_feature_modeling_candidate_refined.csv",
    OUTPUT_DATA_DIR / "06_feature_redundancy_summary.json",
]
expected_table_files = [
    OUTPUT_TABLE_DIR / "06_feature_redundancy_font_config.csv",
    OUTPUT_TABLE_DIR / "06_feature_redundancy_input_file_summary.csv",
    OUTPUT_TABLE_DIR / "06_feature_redundancy_feature_presence.csv",
    OUTPUT_TABLE_DIR / "06_feature_redundancy_feature_inventory.csv",
    OUTPUT_TABLE_DIR / "06_feature_redundancy_feature_family_summary.csv",
    OUTPUT_TABLE_DIR / "06_feature_redundancy_structural_duplicates.csv",
    OUTPUT_TABLE_DIR / "06_feature_redundancy_high_corr_pairs.csv",
    OUTPUT_TABLE_DIR / "06_feature_redundancy_clusters.csv",
    OUTPUT_TABLE_DIR / "06_feature_redundancy_cluster_summary.csv",
    OUTPUT_TABLE_DIR / "06_feature_vif_candidates.csv",
    OUTPUT_TABLE_DIR / "06_feature_multicollinearity_related_features.csv",
    OUTPUT_TABLE_DIR / "06_feature_multicollinearity_summary.csv",
    OUTPUT_TABLE_DIR / "06_feature_redundancy_high_corr_summary.csv",
    OUTPUT_TABLE_DIR / "06_feature_redundancy_vif_summary.csv",
    OUTPUT_TABLE_DIR / "06_feature_redundancy_action_summary.csv",
    OUTPUT_TABLE_DIR / "06_feature_redundancy_decision_notes.csv",
]
expected_figure_files = [
    OUTPUT_FIGURE_DIR / "06_feature_redundancy_top_high_corr_pairs.png",
    OUTPUT_FIGURE_DIR / "06_feature_redundancy_top_vif.png",
    OUTPUT_FIGURE_DIR / "06_feature_redundancy_top_corr_heatmap.png",
    OUTPUT_FIGURE_DIR / "06_feature_redundancy_cluster_sizes.png",
    OUTPUT_FIGURE_DIR / "06_feature_redundancy_high_vif_related_heatmap.png",
    OUTPUT_FIGURE_DIR / "06_feature_redundancy_action_counts.png",
    OUTPUT_FIGURE_DIR / "06_feature_redundancy_top_pair_details.png",
]

final_checks = pd.DataFrame([
    {"check": "project_root_is_repo_root", "value": str(PROJECT_ROOT), "pass": (PROJECT_ROOT / ".git").exists()},
    {"check": "data_root_is_repo_data", "value": str(DATA_ROOT), "pass": DATA_ROOT.exists()},
    {"check": "reports_dir_is_park_reports", "value": str(REPORTS_DIR), "pass": REPORTS_DIR == WORK_ROOT / "reports"},
    {"check": "input_05_data_dir_is_reports_data_05", "value": str(INPUT_05_DATA_DIR), "pass": INPUT_05_DATA_DIR == REPORTS_DIR / "data" / "05_content_feature_engineering"},
    {"check": "input_06_data_dir_is_reports_data_06", "value": str(INPUT_06_DATA_DIR), "pass": INPUT_06_DATA_DIR == REPORTS_DIR / "data" / "06_significance_tests_and_eda"},
    {"check": "output_data_dir_under_reports_data_redundancy", "value": str(OUTPUT_DATA_DIR), "pass": OUTPUT_DATA_DIR == REPORTS_DIR / "data" / NOTEBOOK_ID},
    {"check": "output_table_dir_under_reports_tables_redundancy", "value": str(OUTPUT_TABLE_DIR), "pass": OUTPUT_TABLE_DIR == REPORTS_DIR / "tables" / NOTEBOOK_ID},
    {"check": "output_figure_dir_under_reports_figures_redundancy", "value": str(OUTPUT_FIGURE_DIR), "pass": OUTPUT_FIGURE_DIR == REPORTS_DIR / "figures" / NOTEBOOK_ID},
    {"check": "modeling_rows_is_14922", "value": len(modeling), "pass": len(modeling) == 14922},
    {"check": "membership_row_id_unique", "value": bool(modeling[ID_COL].is_unique), "pass": bool(modeling[ID_COL].is_unique)},
    {"check": "target_is_binary_0_1", "value": sorted(modeling[TARGET_COL].dropna().unique().tolist()), "pass": sorted(modeling[TARGET_COL].dropna().unique().tolist()) == [0, 1]},
    {"check": "candidate_features_from_06_gt_zero", "value": len(candidate_feature_names), "pass": len(candidate_feature_names) > 0},
    {"check": "audit_feature_count_ge_candidate_count", "value": len(feature_pool), "pass": len(feature_pool) >= len(candidate_feature_names)},
    {"check": "high_corr_pairs_file_created", "value": str(OUTPUT_TABLE_DIR / "06_feature_redundancy_high_corr_pairs.csv"), "pass": (OUTPUT_TABLE_DIR / "06_feature_redundancy_high_corr_pairs.csv").exists()},
    {"check": "clusters_file_created", "value": str(OUTPUT_TABLE_DIR / "06_feature_redundancy_clusters.csv"), "pass": (OUTPUT_TABLE_DIR / "06_feature_redundancy_clusters.csv").exists()},
    {"check": "vif_candidates_file_created", "value": str(OUTPUT_TABLE_DIR / "06_feature_vif_candidates.csv"), "pass": (OUTPUT_TABLE_DIR / "06_feature_vif_candidates.csv").exists()},
    {"check": "multicollinearity_related_file_created", "value": str(OUTPUT_TABLE_DIR / "06_feature_multicollinearity_related_features.csv"), "pass": (OUTPUT_TABLE_DIR / "06_feature_multicollinearity_related_features.csv").exists()},
    {"check": "refined_candidate_file_created", "value": str(OUTPUT_DATA_DIR / "06_feature_modeling_candidate_refined.csv"), "pass": (OUTPUT_DATA_DIR / "06_feature_modeling_candidate_refined.csv").exists()},
    {"check": "summary_json_created", "value": str(OUTPUT_DATA_DIR / "06_feature_redundancy_summary.json"), "pass": (OUTPUT_DATA_DIR / "06_feature_redundancy_summary.json").exists()},
    {"check": "vif_rows_gt_zero_if_enabled", "value": len(vif_candidates), "pass": (not RUN_VIF) or len(vif_candidates) > 0},
    {"check": "multicollinearity_related_rows_gt_zero_if_vif_warn_high_exists", "value": len(multicollinearity_related), "pass": (len(vif_focus) == 0) or len(multicollinearity_related) > 0},
    {"check": "data_outputs_exist", "value": [p.name for p in expected_data_files if p.exists()], "pass": all(p.exists() for p in expected_data_files)},
    {"check": "table_outputs_exist", "value": [p.name for p in expected_table_files if p.exists()], "pass": all(p.exists() for p in expected_table_files)},
    {"check": "figure_outputs_exist_if_enabled", "value": [p.name for p in expected_figure_files if p.exists()], "pass": (not RUN_FIGURES) or all(p.exists() for p in expected_figure_files)},
])

final_checks.to_csv(OUTPUT_TABLE_DIR / "06_feature_redundancy_final_checks.csv", index=False, encoding="utf-8-sig")
display(final_checks)

if not final_checks["pass"].all():
    failed = final_checks.loc[~final_checks["pass"]]
    raise AssertionError(f"06_feature_redundancy_audit 최종 검산 실패:\n{failed}")

print("06_feature_redundancy_audit passed final checks.")

,check,value,pass
0,project_root_is_repo_root,c:\Code\ott-churn-prediction,True
1,data_root_is_repo_data,c:\Code\ott-churn-prediction\_data,True
2,reports_dir_is_park_reports,c:\Code\ott-churn-prediction\park.ingyeom\reports,True
3,input_05_data_dir_is_reports_data_05,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True
4,input_06_data_dir_is_reports_data_06,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True
5,output_data_dir_under_reports_data_redundancy,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True
6,output_table_dir_under_reports_tables_redundancy,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True
7,output_figure_dir_under_reports_figures_redund...,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True
8,modeling_rows_is_14922,14922,True
9,membership_row_id_unique,True,True


06_feature_redundancy_audit passed final checks.
